# Đề 12 – Kaggle T4 v2: DINOv3 + SAM 3.1 (fp16) region / re-rank top-20, 70 video. Built with DINOv3.
Trọng số offline từ dataset private `holilihu/de7-weights`. Toàn bộ kết quả → 1 file `/kaggle/working/de12_results.json`; dữ liệu làm việc ở /tmp.

In [ ]:
import os, sys, io, base64, tarfile, json, time, subprocess, traceback, glob
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DE12_FP16='1')
OUT='/kaggle/working/de12_results.json'; W0='/tmp/de12'; os.makedirs(W0, exist_ok=True); os.chdir(W0)
R={'started':time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), 'precision':'fp16 autocast (DINOv3 + SAM 3.1)', 'stages':{}, 'errors':[]}
def save():
    json.dump(R, open(OUT+'.tmp','w'), indent=1); os.replace(OUT+'.tmp', OUT)
def run(name, cmd, timeout=3000):
    t=time.time()
    try:
        p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=W0)
        R['stages'][name]={'rc':p.returncode,'sec':round(time.time()-t,1),'tail':(p.stdout+p.stderr)[-3000:]}
        if p.returncode: R['errors'].append(name)
    except subprocess.TimeoutExpired as e:
        R['stages'][name]={'rc':'timeout','sec':round(time.time()-t,1)}; R['errors'].append(name+':timeout')
    save()
CODE = 'H4sIAB9Z32oC/+1963rbxpJgfvMpepDMMSCRIMGbZCrMxrck3lysSE5yzkdzcEASJGGRAASAlDSy9tun2QfbJ9m6dONG0nIyjid7RHyfRLLR1+rq6qrqqmqzbta/PnWuv3OdiRt99qc8DX52fTYarXb2HdOtRtNqfiauP/sEzypOnAia/+xhPs1jsUy8pdu3jh43j7rHnU7TbLeanXbruPLZ/vmXf9ax60Tjef3PbAMX9VGng5/WUaeR/0zXvNVpdrtdSO3Q+j9qHn0mOp9y/Y+C95Ob+97/f/qYfwn639qk/9ae/n8S+n+Up/+d467VMTtH7eMja0/+HxL9t23P9xLbNsObP2X9d9vtD6D/HavTwvXfajRbn4nGp1z/e/q/5/8fOP8Pq67ZbDw2H1vHjx8/bu83gAdE/8dB5P4ZtP9++t+02i2L6P9Rt9NuN5qw/ptNSNrT/0/waJr2qzdxA+EtnZlbD0Zv3XEiGCl6Yho5SzcWta+EPl0tFjX6LQ5F5M68wDeEuxy5k4nnzyjPN09enp/jl9idLV0/iU2oveItwyBKxNs48KsiiKtitghGVYFER70br5tV4a+W4Y1wYuGHVTF1vBiyJgF0ozKNgqU4ffmDkNlfYlc5NYkcP54G0dKNYvX6ySoJfgwm7qJKX0+jYOzGcRDxTyqcplUqv758/uKV/eLvr0Vf6Jq5DNtaVWims/bocxms+fOCP6/c0VIzKpXPRa30pMAqPZWJOxXO3Inn+jQyehUBzwwag0GbkRt7/+nq+HW8Tp4FiyCCTFV69+zVD6/O7KffnjW/PXvyD6Mq9OOqOIZPz0/cKAwWTgJz0MesL396/eLMfnL24olB1Udusop8oc/EV2JmLl3H1w3DnEKJxIWvFepS7CzDhWtzr/XQSeYA9jDuW2ajKibuZBX2G7K7MI2/+B6CmUvBhJuc5atGT8QXXigcHr64mgexK7rt2shLhPMdjFp4sbjykrnni39SmX8KeBeLYCqSuSsWTpyICzdMuAJqTweMdMQoiKLgyp0Imunn3uyX0DDFPzx3MYmFntixO64iWpg0pQYhG5amGvvip8Dn2sZOKMFNmP7MCQE6Lo3YOBFrGDO+dkJz5iY0Fc+enNqnZ69O7W9Ozw0RRKLZMRtUVZy4gJwe5F8617pVFVGw8ic61VFH6BkwP5z1au4tXPE6WrkMQ3yCCwBxJFuLgOPSjfSdNxV+kECWnhjBq4v8C0/8OzUt+n3RyKrDZw61pciFeQnEwl3ALKQQyFXFrx1/wmCCucFWMSclAm7p+lz8W59eG2a8WgLqiC/7XLDYNlUpDvvCOhHjwE88f1VsT87EvJB4gxMI5eoE+iqvZhPn2Iki5+a+xXD27VMjBzVqXk0zwHThOrGrb1+gTLV2rNBZ5E1smUP3lrOqWLhrAGNfB9rUgmkN1m60cGBRmM1Oti6euwkQzCAC0ui6sgURRkEYxM4i7qlSISwZsVwtEq8Wjx1ADGzOFGe0UGOx8ABQsCD060b1plG9tqo3VobQv1XFdwBG6JSJ5OIEVsa1i0g7GNJ7WJjCh6mTPc4maXwF0EMM+Q2g7WMt8HEiQkgOMXl8JQ5UDyln+iutAuuOsG4gtDNX940iCuDr8e7X+FBnTQCBCytFx3XTIBqmjw/GV7XwCpdMLjU6GM9r4ZxSPV//jVP18aFlQP5DmR/efCffRPRmfohljAL9o5YrlTGgYSxe/fbDunnG85vO3qsQaaizEAH0rrYOxs5IboE+bA/ZPAr929NfoNpxsISNDZYCEKJZEMwWsGNeLdbN2giwrgYUZTy3ujXXj2FjXLjpBCJ+KQFbj93FFHY7oHV97f46NNwEw4v+MdLktTeGQuNwpeXgvHMffIW15nY/+v1NEL2iATLiwujTirBjJpLKYkFamnYIMI0cz3cnOnYdCCdlX6rsm/VuL2cmgc4DMUx37SxyFJAqBApJn5AHqsahq4FTvq+JIzD9wJ5FKflk8MKyWqTghbWSA9E1VMWj04nNifu0wBlNbNgTYZhxXwsTjfqnOpB1bbqkPYSHbFIdNvM+bqSH3rW7sGEsK6j4eqDlf2tDY9AYpvUA3wO7LCy/CdSGlZrx3AlhRU9dJ4lVGnAFmKpj7gPMnHUEkDPrR4apCOaJh3RIp5qAcHuzZeABhArNS7ohy8PPcsEq9QALCfG5GF+Pb67muLxDB4Y6qcWXKydygdgHEezBg0bVyio/l7uiIlOAIlGeSily8Rbrg66bOLkM6wvD9HzoBtCJJEBaqJfoyPgayNNNVYwAGiOkXDoNZfB2CCTr3EhLFQoBLRVATAVQU3Fjyd7B7/F1bXRVb2ZkZ3xTG805AeiK6j/AAPMeqrz5Vxa+ujmkUuUd9tqqXTeA8bK6tKPeWLUb/tlDeKSEEGnXdYPZOf1Gfbm2VIpl5LY5SdAiJGdbNjaFips7G9O+cw9ZveeenzHDKQl8jQtq7cWwYnkPcy9XHiAv8O+4Iz1/+dOrdetX73WRb8Y158B2FtOgACSi2Wxfwx/wdsAegvggN/af3EQgA1qPk8n76CHW0YfyuYkn1KC6+4L3PcZbqA1S/NBkfmHQMNvHHWC7zHanSx+N7lDRJ2i1lLfZfIyZoCn+6AzfQ0SQVmynE0XiAg2AGDm+0Ac6thVza97SBLYINtRE14BtAeIi+X09HVo1G6WhWKGnL394+dOLJ2fAd9WB9QTeU9SyoWNiOjRcT94SFxT3dbiBMrdFctRjkcrksejXJm0csMm5uBBaVQFrBbF9ktyEbp/zTheBk7Saxp3aSl9IdEtxaLzwwh7toY5Xxx+1tZfk9rNWU7yDKZzBm2ZPbZzyd3HbQwx6J0CiDNaQc+qM3VEQXNQ5oRYvYXpYQqCUlnBNkESK2VrYeAx1qa2ntlhPrO7x4yVyuzMncSc94bsuELDvvgFwXLg+SLQLoD6w5QpnPAZZxPHHOYFi5/b9vhFr6YaNzHUZrbE84g18nOQ3PC6CMoeujVcTR0OKwtOAP00vtp214y0cYA10g9l8ZgiK9eO+nMrBu3fidKfbvhcDUYZ6NAQrdURXX33qubkA8SzSSUTK3oAUBXJmpMtNBlbA1IPNVltiV2zEKxSkNUP1fr3Q8rJOrmXVdHEvSKKbXsarbIr020dbqMK9xkmG5YZ80guQMqNcjVuIJQwRdkMmhcgUShrISJRkBBS2pYjwKSDmM2sUR1rq9If098O5HSZUo7gP6zQnb66Szf3Xy9j1Bgo5vuSJDCxf2ne3MU74MfB63uEoHm6njSdU7hYk2XUByah54OfW2INr00vcZawDWSltoPdigCB9glwWDsByDHKmXC9bSJcFuwKsU1gxk76erTWom1cY7tJBbLr+2ouAbUUtgPb8hdUE8d/qIq5amkHZ4XOLfDNPGar7OEET5WF77gHt9G3YLxJ3ozIfJEiob9ca8ldLklHjxI1sIl0xdLBBK7CRw9G5E4u2UDmZyMUbjU2Jwyba4iT6YD7oQV0wp/TFAqKIvekNWYFkGfCmZtFKePbDObyljZioXXFxIa5vbUuOBgZnMwuNLOcqQu3T+wG3qXVgZQksOj8mYq1PpbLQfE2YaPSoRUDcIFgArGAlhKukUA8kKWZMLjMQWVb+mCVCWHQR7Da4W08ZjXTcFr1lH2BgAsXVDZM0lvommwYsAMwYwBSwLnF1aMgAngDRUtfkZrpDhQhI715v10+MVt5iYlMGfY16LHviAZMFlfMXIq/3bkmpgk9qOkAEjbwJvFgEs34YAedZ1v3Bylg6FyAkRLGeNuZeA2LZwUUf9Vuw3oEBBWgrvkCnvjBYYKaU2gWXUOwsW1pJdpVHECa+UxoYJcOeP/mxJYX2ExLHkUBnaZIkFlFOZssL/FLk7EM/mRKVukUCuMYbkp7XBJUzErQy7Ro3v3bHQHyXsGaBMUBEbBDVrdJfg1XdJv6T3UUySDQQEBW4EX1KKVPaPBfByMR/OsAdNZTm2wAEj9yEawe4dUK3pmr7Bco1iZEi6qkuO0+nsO6EJGJqsqD1XRNGyCkvy1wARhwWjgUK43BuNeoIcJGqd4heuF/pa1gdWgJvWCOasM5MQzIOiRq1h7QUtkX4jbC7G5YJP07cFsoB/R+RDAqv9aJgv9FftaJRUsMEfWQAei5VcjaAJYh3A/lrKLvOaQn9Vh3nyc96ProrUiOce1U7YJdOvaA2AR1QObk8AZzItJR/Z2khTx+wCigwwb06hyrA9CeNFF9oPwQg6BkoZddy4PAmcUHa8XjDBxEYi7pAr9wIW8S+EQ4BqaWBDhG58dtQ7Z8akIJuWyvop4lXmMQGq6E3Vb7eBPd9OroxXyKhevn8RyfUcwnfLJzk5an+d1Z7gBiNkIJipjOZ2IjCNtSv/30A/6En2FaGxlTLVQRcg6SCUA7QV7uVVOmufosjuDMpp+w57rIIk1tmP3GS+WRIA7yHX4T9EpL4W36DNFoSkEKfVYnBmAVXuLYmpS/+RKAQPCGV+oXnEuk6mMBCQJFeFrdD2Ivy7/1pHbM0JReEh2TmBLYV/fbggPqOjDPUjh2Hj7sqSVl6ftSYbmJBxIkrTW5HQM91qoDUMLQt0c+t2w4T4J3aA3qrxL0twhAPmzcGuQVsFXpQmKARwh440dVA0tL5oRipghGXkWQdJgOGxbBYLW8+qQ4EMD7diNJiNPnDknzD+Ip8KiMXnshI3Cr26/YixSpejDtWIi4SRRlpi4x3VwTStAIngz/TXPJm3k/rv+hbDVID+/1mo4E7TQTM08wJCcpVcQnYWAY4yK5YiTzCBZlaHk28A8EEepeePIAgAJt7emILNJ5oYhVPxZMqULVq6DoXdlKN0UKgCkTwbpgKxvhcAgixAzh2/swdKdFGqSZJp20EJnbk0rHQ7V1hj0op3GADsEOsHIdDezAOQO7BA0wcljaE51XxUqEMTPGA6JopgXzJmjwEZrWcxU+CxFkYxsYJB6xC0lr+pxfqz0kv+BL+v1cWADwdvB2eiAv3BtWV89xeM6c9ZitTi7mJsfUZSNi2+Iq+D+DdEFrtZb+wYmZNUXiDanGLylU8I6V1ii0ExOxHeZoss6NUS7BikDwOy4wtl1a4omNHqtiKMehdDPlUeWeWVFn0I+YQIPsBgwdC7I1wJm+dMaob53gsrMeIr4yE4t0kefdlH4ujUjRI0fREfROElqzbxfL8O0XQ0Y29zjANpxJZngQP01FcNhSglSzaoxKAK4nUb2L2AUhuSmsLg6GCcndxZ6VjOMg+j6lWrEfVmsJxHpvI8+mwCMarKH9InePUsAGsYh73yhwSFkJxNQHmAL4PNJc27y/7COJNdMxlAdZiG8IhdlEugpsG2AW/zFU4QTaB0vrQHSYBfegbYFh/ZHyAvMed7RGIFPQgwdjWR9qfFWMGENSI+KCuEr67xIfhN+6F/MH97SGwUs4sR923NKp2Qea5MUsV11t/4SxHE0dc90TtOoWD8VCM3/b2n3v7z5z/V6f9+Mg8anStbmtv//mQ7D8jFznBP8cC9B77/2a725b2/92jxhHZf3aOunv7z09k/8k6Mtgfa4gCaKiEpnlBWPs+5bd6xJu4DgiJY+A/PGQP1EvUJdZQmBPIvuEmzTJHFbXYQhqUppY0Ff38yY+ihSck357+ckJ2UPjj2ekvaPuJ2txJzfMNKUrKNlGjxzr/uSuIQXT9MbD+IHEiOxS5zAbSW8kYOnEFj/eDyJt5vrOgw34bK7LHQeyBrJSTgdwacwWZGHSImhg0c8xZsKLtKpLK6oa96g4LVWKISagkgEgtHltaJpIr3GGcmGVFVhHtRMq2iaevzu0fz188g6rEgbBgHUFOMjAs2BeelOzicpzQ+w3vttrdkYB7kbMulENExNFjqSeoipy6ksRZnn03UoItSXDAfYGwDRItfglWCQm5ZVGWVKk5pdjJB8mb3I+czJnn1tPXUiJkGRo7xj2SnbiEtnwbe46FMkUccfdYVV6LDTly87xLdRunQqBBPyRDOywo2aAyGBF2aouGTRnwKICi0WfWzcM+aaMo04ZhKaVuqfJFHiQEsWmkVKeZ7pVKk1WFt8QOvBBfi0uQRk8E2iOhwQq+MJ1ohovOKLSOb9BS5ysccypnxEUpQ8qxnNVgQYNsepShj1GVeOZO+NQhnVMWqd7Dyue5f8gPwiriG0jitxpXKVVyZeUryhyKbqHij8F89y/FF+35/z3/v/f/2vP/+cPXjy0F3MP/t4/aZf+vVvfoaM//fyL+nxhy09pwXmCj98Bf3JDHAbHo0coXV8Dzvnr1Y+3CWyyAZ9b/15HZFN8+FWfn53ia5ydkIU+8w/fObLZwkdMHdvsnNJfqiZmXiPECuTVltwfse4aE4m9/E6EXCrLsWCxEzRVmtfLdN/brV9+/+ImlADTSi9HyKjP9o6L14m/TMsWT05cgk0QwuDAQZy+ePP/xhWBbJj4JBE7/HLJmhq3A+bvXJLAsw8QwK7/EkLdXML8wTTNnQEEmDQa6vTF3BsIT9ltZpsQgeCTjOYpVKJ5wvShJzEFikodWExQjJPj5GBvFjh1ChTyUy4wgdp7Lybb6usZSmFY1Mm8CkG5sZr0aZqtgq4ieIO74IgTuNSERpHy2RB0jALNNH8HGjVQ/GVREUNjMh3LtKm3mMmamdcr8Iz81GU/JL8lq50SZTH4U28lU0jspwwCFqlIKNFQ2G8NZsZ99f/o6VyejWX8HWPS8Vwea4ZAUESzt777plxuUckHpFJOKeP40KB6poQK+WL6HpltXQXQRsyCDUxAmJEDzarHJQSlcuNeQDuywc+GKRxPp4GQePEJ1OZlDE5CWXrzcMP4aX6Q2ZXTIW+oCSuGhvQjG7LsoB33lerN5EttIapRFEVU0vmCw8vEt4OVF8fwgntD5LYi34QIWvq6lnSUTUzx674l1wewQqpSnMnTeZtJRA1vP5EqXjBJRvmNkJWCT6Z6NbhN6PAFJLonga/8bIJklO9PN+dEAGsk2+5kSoAyyQODu2Ah2aWkQT/ANgD4GgiLTIlP+poz4fuW71yGUdSdpliwpzaVKsUkQmiQUKhr0OsMiGMg+jJGcELWnVUtDRDMWQPYxevFMNlYAg/AeByCkAWjklV/50qDsJM3BJLTgMERLNKOu5DiU0rj3uAn1pB6EnUh7W4kM+y2j6Axbw8IbZbQstLqZEeUWIvM7bEsrBeNW/cOtW9kaDHtCJC3rpumvFgv5UzcMZSEJE+VGLmxNRIHKrjapoAwwJWCRyVUGQEra5mRFqyK1F4apQ50VkzqqYoc7UMj6lGxee2UrzY06cTw25yYbF7dP/9WG1w83T/BHVVbc4Ak+VElH5Whaqkw7Uw8isqiUuoNt743ejmPUvkLDopsP4uIKlhf5+0gTaCJEI8MgW9rrm+sbQfauwHCt3cib3pCBNhDZDcPSwYjK2/KAWB5lRuyjAmVjl8nnQNqaXwyH9wlx+/hf+/hf+fiP3UbXfNw+tjr78I8PSv6XPNifcgB4j/zfaVvdNP6j1TrC87+j7l7+/1Ty/6vpFF02FRvOApqU3CdOAhwqmmX5AbrLTYIrH7m9GOT581WILBJu2DV4V4uTGygxDVAWhALstELmnCBUH4CcN3WlmxDyRTdXc+BCxMpHybF+Qc3VPZ+8NGoCY5WIqbcAjgHZ44l49CXxgF/Z9peY/NUjoZPv3xz25YU3X9Un7lFNjoCE8citgZxMPvQTNh5TjaAIBExu/aou66yLteeI+GYJYLiIsfk4XKCWIuXJY2gf+/EVMOdRYv/0EzR/KAIVPUC9jOdOs9M1k+vkEfUhZ+W9uw9UFASuijqORMGCmUDp5Fgn+cyyUXaQP9R39PNAH5x5zHZhGFAlmM1cDO9xgx4uufNLjruDnCzH4sGoIcAlAmc0XyXeAnkIPs+beMD+ADuh03TEfa0wPSiwQe+z1HQsRTcS5mUQJsyUMOPEc9pXvEuY8YCZ6wPZzGLLd/WDg/qBhj0br6LYW0v+Jm9n68VYpR7KMyfmF281Hj4INQO0lZZ5I3eBnzrGcKEGjDuhp2+BOSfv3NCoNw8Omo2eaU3vxI9PDS3rJHV/mJkTLsiZHTlG2eCwR6bXmhAIDWPDhQaBVfafoUyfM85v4HmlwChT+xkLiobPG6JkWDh902yb3DJHRcZ1yQfFI5NQXcdcKC6fcGQC7Rb7eVe/Xd5pJ4X+T7Z2vhzEZsMGegJ1Te80gyRfudJwGnKvJBQ2lh6l47KLi5ac2+CxpINYk/QSeqTp5qHBK1Z/Mzk0vtCq26BVANeyx23lbUDTA12AgCwDsqU5i4JVqFsoWOVMRMPMxweBRc6cIJnEHEUBa96wCp2QOXQGdCxzBxRBq7wfrlCuJI4Q1SDD+gna3WpXIxAvnVhcbXerCXNSRBjD5DAlMMdBeIOQDUZv2UwfHSAiqItiV1jiyy9Fs23k3UxxlVG3iTDZkXNFNqfQi0w+nKMhf2GVT9RgJeU8KPqbYAmkahklKY3W8RFw1EN4iSbZ0upBYjXGsTBRPRNCkvSWOiEjbUn7TG63FDiiDMVIQnG6wzlpQf4AMK2RzifQPTHljihQgcCmIfLP1aE3lDHKcm4GQ+4VLDECoj435+71xJu5KICitgAHDgNBQqPK4NxhIQHr8pYqe1Su7NFQrbL5NKPAt5vIfT9Nzu3tW6jzYbEcY3Wx0KYdME74Du+3bctPKzAUmmFIv6zQI82htAQ4uEBX/2tYwnweQQ4GMHqPdPQ6uUVNUjd2zEFJF7imM1Ur7qNy3nhH7mMr7Ay/bqEWhyMOaOSHW3hHPqP4zR5tKzTCQlXaxeVr/JqWaubLNJVrDw1wEJaIIDnQZa6AuoYEBCPWhckcSg5hVgZFVSgOq6ALTQtr2So2hgoEAy3jQggvGbC5fkDHyF1SWdZEVGJg2sP/Yb2jwu9SDfN2UqymYgP+3Pi9bb+nytT9FOFSqF4qV6CVByjy7u0/9vYfefvvjoXx7Y4bR+3OXgH0AJ54DNxREv9F7v/oHlmNLt3/0e3u7/94OPR/r///i+j/O22rZXaOm0edx3vy/5DoPwZgmHggKzsU2OCjngPco/9vAc4p+t89aiP9b3f29398Mv3/Dw46ZI9cfzxfOhFG7XIW2U+hWw126Iir4nLlRh7Imit4iRHEJijs48nzjXj+5NeX56IJMysOKJLZgYhdyO+PXdSdUMTvK7+S4RhXKvRug3TlR6oVUzzP8sTBKkIbOjx8Jx26bOf81S9nz16cm8sJHjSEHrxLAhXELy5g8nJilL14WBOu1N7kzcMq8citxNGYbACq4jnFvnMSpz5x1l5sE0hQpC6n2QuEYPqGeqhV8lpTjtdxV+cRahs61C3xf6AIdEUVqR9QZHqp7pM62lyV9dsN0XdqSN1tQWVoSLM1pYNV751RTKrxqUHxZrFDl9ihgSbnHIcnv9pzJ5pow82+3F7+oSblSLF01vrb7eBQXTiQeqSCzlJ/S1NnVADzYqUZhJLP7+pk3nPuAqVDFK0ThuJBjVZWGZ4IaGc5KZbO0C3Nf4LYxuYsObegS6mMXm/OUIaT9m18R9NZ2a3bXRsl/yKpritABHr2P09ffEtji+vt40ZYv0UFnfk2nFEIlaso71z2W0Q6ynW1nGRPcZmN9QMNerXGScDoqPpxB/5DrTkNeaZzikrBCZayJW9J8AmpdY7Xo2OIUkDvpQz702tSvCEdqq4KaETaYuYuRPCWheaz9qHKoi+b1PgrjdNUe3MgbiHBjccYQjk27t7Eh28GPwDivRm+0fXBfxjDQ+ONgaZrOM88j0p5P9Uw+Gh5nsQ7Ad/wf6b0p7MC7vej2HU3yNIjyK8ZFcIhDUaFsXLrO+iTDNwjgaV9nieAjDxCz5FXwl3jjf/EB7whQ8q4J549E0//IdpmQ2UNopnjAywjPKr9hqIP9WSs8DkaybJbpCSwqhlJZkeKnOv/CFavVyO3fuqigVIdcWYd1P/v//4/5I5Fw/BGq0IZihgvLboBB1wxCQjB89kVqX/jv/HfyY3gXbZfvNvo3zvIV6vV0r83vgY7jwYfrCIGyBoywaiwnSLaPRZWS34xSnJqkBqWaLLxMPndvf5vr//L3f/4+Oi4abasRtdqdfcC4AOS/3L+LR/dBuwe+c9qNjrq/scWy3+t7v7+r0/zSKkIBJjQiWKMR48uHugAA1+khQ2wLFGCwaLLB7G7fivO3rbxaNK26WKWQqxXCtdQ8Bci1KtQNATVGfNJNFthPIZT/EVn905IoSId+ULXarVUporw5oJIuYdvzRuQ+VIxY2VLPuVwIi1A7gmru70tjKBWFWSuT0bkWW2W2djaqgpDmWtXRucdzwO8HaM/0PzAJ0lTvpCRa6sysu5we1c4lqXsjIzxyw3ApM7dRdjX+KquubNcsqca3kESLCZV0egH0ykNEWcmNGlmsHYMvpv3inNMpSVwTJKfHVPG2nRMCrPpmGl0TQ4x7Jgcd3a/0ez5vz3/91fg/2T8r2ZzH//rgfF/6Iq3chL3v+H+706n0yjF/2pZnc6e//tE+n+pUiWFag+DvJPPntYT7LGrzciMWkXorMronFWKzHk3vBua4oyDCNMthjLgbBZLy6w8SYNyeejmvgBcw+9T4SXqvsEY2ItvX/PVpnj8oB/WayIJFgapjm7wEilktjDcAEUEg7wqIaC7eipn5JUcizMXPRS/vrWqnarVuBO649/kGvU5shka8D45/dqiaMqOryJgL9Baf3yTPy/IOGM+IpD8sQoFlosC9iexzOUIACdiJyetInX/ETaamLgP46KzE4EP46QLHC3HhmNOsEqBwLY3ApO/g31umJ3d3P32Nxj7zYlcZzdDrvhg9EFXQevUaRfZAy+d+EJgHRjdji4JEV8Cn+zFO5rkyFL5kWuqDeTdyR6yp4LuEVI2G9xSOQ6Hdo/ckzWwg0v/GY8pLkV6plOKwe6YcqAc3+2SvZRxqBjPbMwXzHyJ7LyE4vCELvtT2AYVEPJgZG8Qi3IuCbuQPNei8gq+vSsKiAw+hdj8q4KB71TQZDThVbnSmzmkj8mu9ZG/2eNEyNryiRV3Ua43d2PJ77utRFZfvK3k51+r4ufX6fUg2Tnbz9wE3tLRlx7pNDkFQ2gEb1VcDiSFHhpG6YK5E5GU4+X9/Ks63DjPBcTDF6/T62+Kcc/4/I3CrUPHWIqLlZ9IVZ1BnuGkWT0aSIc/rAZ+3p2IJ6fs6H2CBFV+O/2+GCn70qvyyLNrKX7OnXmVxpF3C1DT0yvFWsCAzjYdB1E4wnOUOJVIynFNcEpUqD+MO5hGHPz518GlNyzd17VRPc7mtqiBO6uB0SsY//waX3FcySKwVeZV7E6Qoi7I351c0Uuu8ltCD6ojMLaHTu/7mBUBCxgzwziDfJe0DDCP7ZF7xSwNS4gInwUp5MggA7nfc5RxFU78EGALFFpl4cSvKIPKXuMsBVPrzJ99oQCzzAdwBLTEbiGt05fFU0fysTjribPBxTA9riPnfNjgdagQY9DnjwqTgK9sncUUp/D0+81SxZFDgeLYZ4Wx02sVsrEADOweQVzBOdcLimnfFw0k4YicDXmTeHobW5WvV84mC4ZibAR+j3pcE1+2jVXBN0qpC5j2Q2FlTcLqU0PFjHW68ABPw1T3EGPTm5JjvkMBV/PBwe1UO/v69uIuvZ+EYeWHfAfYGg/MWsV73M7Se9wwpAjxVDtKQ7+4PF63E4SWTcD0fPvb1ztKnH6fFljGdPR/k+bEiKcHxeyw3DA/3bfi24pL4QgoP8uQLnw2yOs2HbzBuwnwD738FSx8nwtUcG5O8PRX3cRCtxfR7RR0rEvXxsTqahap/MIDXb4ZxQf+4q+r4trbf+7tPzfiP3T38R8emv5nFq7YmO1PUADdZ//ZanVK8R9arcb+/O9T6X8wyiPsn/Eqcjn4uS6DP5zCHgvsw+u20VMXi3LAQnlEFKxm8xA3Ow7pfuHe8GVTUoAFllRJynHoAnNboeDwv55Bbgy4kIuahZLl0l0G0Y3tLCg8HDCBgozTUGPE1lMZitIeLc5cZ4H6lxGGm0C5Pa+2YWUN62ikuWequMF2K5/seFNdRlYt3r1YwegUKsDYtthhyMTQBcMwP1qFr4fOZcZbVDmOmk2dw/DvJJNpACe6t25NfBYUSG/9tm0MEgUTYtt3lTXfFrhhV7jDvLVsOYXh4So8FL4hMgttRTXLQFV8O/nGrZN4BApM+dkAGDV5sx6yoMip8U+DIlEgwuCtTNJFU94t+X680ena9IODVgPv1avkcqNVWsJcpyxEV+KhlOzmry/dCNgGuP9ryz598vo7PGn90OvGkQHksG3YgBzXoNceFjoV3wCIoSk0e9wmv6uCdLtzt41ZdhSuADSls7H0RyauXgOBn+8dTHnnHJhRfiiLpJKFJkCtIZs9G6E2mObiroKx/NxCL9xlmNzYY2c85yF8AMQrv0eXUtkAChQoq1YImYpOwu+JzpnlTmMXUhFIyqIZljIxikocLIKMQ2dAVjXLjd83yT4G8ohXS5oZqsYwskth5VS9f+azwKJy2vmySb5KT038xlTD9OeQQcpaFBtvS2noZSl78z14wpJWKk7F+llOHjKKUUVSUWrzauFMHjuTEpa2fUPQ/v8Ruvby3/78/695/t9otEH+O+oetRv78/+HJP8hKSa/HqU3+4hy4H32n92upew/260Oyn+dRrOxl/8+kfz3Hcw63ptcOPTs8c1b0peEX9E1XF4s5tOFF4Z4WoHhAOOxs2D3u07j34FLjTAGn4/x+a8b5hFf0BU65FcCkqBTiSAhWNb8wItd0Wy2r+EPbxVbOyB5gggarEGiiMeun4YyFg7xH+y4Ql0xUuFQ+YNxOmEwi4cbsiCKfvfe2VXlj1dor0jfXvhzjOJfGW11BjwRkT/jK9d5WKY8jLUhnUSyn7feeT3KHNm449mt1xzAbjAs+w/mipD73VYfwuJRojpJlGMwn6ZTo6tRmksvioJIz504clu3l4NHxE1S1KjSMaNhAGNNdeowxYZszkzmIIz7IMDqumU1gSe2mjiiUXakmV02BvAx0dpjBgIpyt7NTqcq9Cb6ndE/ZEQ51jXAdgU5j+WJxWhmEjaxjxiWAEYWGr/yJskc+FmQ/bgiTp5TSEhOx87gmVB2gJpFjpewrZfdHOta2mzsrF0FnfBO47lSJy23BweXeAO8Mp25MyoZ50onAjvnPkPaTRb2RGTOTOjZ+NF52r8G/9fc5P8ae/7vk/B/3YL/z3G33TIfd5vdoz3396D4vxAooRO5Nu1uH/kM4B7+r9u2Uv8fq9NtY/wH62gf/+FT8X9P0YlDOCJe4r1FpOatoS7Tc4uBIIhhyvkhk50mOp0jj5e6ImeeyIZZqYlnWQjkWPyzVkOtEjXxz9TdN+bQyHgjFDpb67B1T0MO4pxjuFj9DDW+KvGpxJXqZJ43wmuqDq3OvwPDiWEHpszEyt4hA8pl0awJbfEmaDLI6jWhT4HRSrCL6PcC6f+UejfY/lVFWUCLyBU/vXot0GMbBofMLzKnytoo5p9ksCCrRyMXDJRBW/u1OyHgnBOXm44Ey0xXMAfFrmHOb1/3UuNYYJNjd2yQLevcLQaZZldqDq2dsctsvEh2Nn6NtLWinsuUDgt4oZt4u/FrFjaaT1M+8ArcP2CIStOdN2vMBdV4n1dXMX+eS99qmJoi4la/qNZuO1IM8rC9TGN7IYlPW8sAh3yf81ghP4bd3WbiWQp44Zg0/o2wF4A9W4NeDHp8loLVbEgeZNGyM3ZJUZGbZc+MhEtySgB8cnyZmUDiOU+K2vp04aAdXzCdSiMo4p1jmZ0sP3OmU9Oi6RQVxkCndBfTMH+bzxTykrFadrsuVuqVQyzncimjtF6ez9ehZ9AC2l1JtzZO0D2QNyxDJaOoEStT1VKQUTon8+iSF4yhg9IPqf0B9gbaLKaYKftPISfIlO0yHqy93toj87s02/BErPEIkEKOMN7fQp56PZ+p12hOOOiIjAqCPTuhzmdGaWRk6JIQSY1m8Hsb7ghBkmLaZiAS9zIfisTx76shF8+iWEXoz7ScSR0SM0IIkAxZmMxJr6GRC1/s+ACbwjEnGlaiQOl7gJ+0adBZDR6G3DbusjYui9FM3oaY60R8VxW/oQCZBTJJS3BuDuGxsQp4KDZpNQgiVXGZNXbhhijzMzY4pqQXjBVvQwSeL89GMWfuvqzLVPpMBc+ptrtJDVI0vBrocsOro8cohHnI4rGHmAG/0MCxp1DcL6D43fCuaKEZeBSgCeBdNF/Ec/frmLUktOmkcEerSyi1EUMeh3odo9F5u9HYcmc1PtewbK6tqriBzxsLar9GZYaP5+b4DW+hhhdp2o1MA4H6uipg1+Jbq02rIw6Efm3B0K4bBoneaeoNpt7kw9986ETDtnsLA7tTcz3AtrGjjVp4Y/SwT99htw/hF91Mjy+v4eU1v/wNx3YIv4alq6yIwAHqj4JgoUs4eikcTTxCNMRXwJ81eBkQmUGsGRajjX8A5pTGUECezDkgf6R8zcfJ3wHsfoNhtY3qZpRvfu7FvjjFPfeOyRLsI4Rd23aKMiamoy4i4nRjUXvDDdyb5oIU/Vs+SFEvLS+DFE23h0iS9Vzl7nHfGoapvLPWbwkId9oHhGeSW8+u5tPAS9NckH5cw7AG1fKtbIRTyumrLqs5ViKP3wV15RZtVVZHvLsOSZO2n9myvgsKpXviHe18KA7cFmP6ULWHj1LLlEcG5OUfJ5w3uISUomK7ym9ifBPn+e+HGQFob/+7t/8t6P+sDsZ/PT6y9grAh6T/I13Fn3L72wfE/zlqZPq/Fq7/Znt///uneXbF/2Hb2Qfj0sxM8Ic6NOe1TeTCnI/QwzZz5B6EpokaGdRBnh2xeS626nk6O1x54+1+t2VvSYrBSIa3yl8xJyJDMbrruHSoqwL24OeFVD2kzFg0eESM1aPh3ZsEfxGj/mhId6TVMAG4dfkzFjoZemMq+8upF8abhK+cpxrwG75p0e1fqinpiX9L/l15e8VaYvTMxvROLB9ssMZ/Wf5vb//338b/Fe3/rPaRZTat4263sY//+JD4v2jlowuFjWd+sGnMP/b6f8/9L1arqe7/7RxZR0d0/ttt7/m/T/F8/m/1kefXR048r3wuvqGYy3Tuu3SSyLvmm3TZ3J5DrosTdFRYOtGNwFuDY76snnMgEk2xCnMRzCruNXFwr8+e/HT+zauzH1+cndu/vjh7+ur85et/9F20O6u4a924FS5wT0Lr98UXlviiKb5oaScivEnm6Dm2GZ9KSM6OT4e9ybUNxdKoNKJ8/Jeqnb5o8gC+aAk6M0y7/cWtdQdvv2jz++ZXf7PEOzGL3FDUXgjtP9jf4R3zc++QmTOgg3eVgahNs04UvIHq6CrO1Q3Fu3fl0RSjrQoVzGWz7zKd+7u9KSHDVYoP9UsSdLwoEGbK5wZjoBTGTTUSG+nTHYmFBvk6whrbC2wNiXkiJgEwkO5afOGLnJXbSSFJ5CLz4BlURxM2VQpFB5itX2p4KP72N3FbqEPVSKo8nJQJnrjt6fqe/9vzf3+A/+u0j47N46N28/HeAvDB8X/k5/uReb8P4f/a3W4W/7HN/v9Wd8//fXr+j66CqsVuwjygnl7KZKC7PfB96JreE/GFBwwE2fVgKAA2aUOTGOao+EJi80MYwBPxtL/d3zzHXf1gI5PxB1mqL55uYaK4xpRzel907xyXhLG3C91i7uTjdozr/CNMXZ6ZI9btZ7rlOeO+UpaJmDMaiGKBGSD2Fz/vHsdODjwFpuLAYWQpzy0rlIEgpTFkkf0utr3ZM+nT/gf7lsLzvb0r9qfYIrOUJKM8+eGH569+evEvtDPu+b89/5fj/9rtTte0up1Oq73n/x4S/8eRM8KVSdvex1//7/P/aDabiv/rNDDdanWs/fnvJ3km3nQK+9/MS4RDSFAnvqM+ccfwGeGGOtqeXuFttvnYcSYt1zS71ngyGTWFRZNdqdVqu2qsHB4e7qz1669FrdVoVI/EIX9AwnjhxLF4HTl+DHzN0o2ec37d980fg8lq4ZYiZlLITDcOFisKP5Uz5mbz7gRYJLdg471pqTh1ncRGQz8KKZvWVa/L4pslxkEQTWJ7XlXfrshqezE1bYzWxIl6pbbNIjJtrZr/ygGe+hxFqHL4uwvqXBLBYdt8om7buowKZWyLOsVXGmpACTRjc4Rbkmh842AZcg0wymjCsYBg8PomSD6gihj6504U7PVtIzQquxA3DGIPJ8oGUSLASGUbKLwth0Tmaet43G2OTPP4qOu6LXcnMm+tYwOtt+ZCBO80Eb/pf4repzLzCxVi7dzz3d0Yjo8OYgDCbZW4dhFFW82qeM/LsmVu0TqYDW4R+p6frwWT4t4m/iauHwdRnMYn+083CmJdtzAqEvpKbKKysYnLv6eS/xpa78JAGPUVSEm67MmWbJ+L6SpK5m4EU6ZoyQSEv/EcrzJQ/liE+/pb2FjQL2zkitiZujvRHnMPcHRDRSxySSa1oxsmt6LvRvss/JMdRsHYjWEIG5i/I5NC/uN26/G0Y5rN8Xgy7Ux3Iv+uajbwf1dGXAJWG5cA/YeffDEpzt/awwB1ZqKIfRoQbN2syJVyDtWeqvrkstA0Df8qgtETnYoANXwPEQOhyrG6F9UcKe/D+I5LiIl0yp8CdffHrp3egkVXDvQYZ39/zf81bH1vj4q4xBqDvjy8Kb7LUYD8dlbKxT3Gcxf6shPb1gug88sREKhNJCu+k7g1bhxN3ceWaYKQMRl1H+/ErVLpDZQqvSdMahI15Y+UnGKwt19/eOqML0bBLjqaOWUBfQPUEWqCJSmwMfpIbsPm6R47YcLXmHk+lLMpXFqfA8I7QLnxpbOgsjJ1xzb+X6zuY2zuW6CBITvX5IZFkfTsqwhdRSKdGZk8ZGw/sAG+lM3QiwROjaqfDo/nqnVEc0Ufv2uusnoyJq8Mr+zVVsBlE1lc89mMfJwVuwlaxjC+5kIwi9vsEotLHykgfiXi98pf3PwLAmEUSy82NQpjG4D2Woe9/m+v//vv0v9ZXcsyW93H3e7xXv/3EPV/yKiPPvItgPf5fxy1Wqn9X9tqof1fs72//++TPCAzneKMyxjuJKyhTOuOE3Tp9sWz0196AoPXkYMBSOLAC9b4GE8mEO9BOgMK1gcItASMyocR4TDs5FlCcUM4Avv2iCGUitjI4oxNx6cYNyV/WXROsJSWasVS5jbJM3UpyYuPm7GlMWze9lb0lGNCDudE0PVe+cr0ZY7XxzzIPGmcoqF+oJDbVGIkfEF1JnwZB/baIS+amJmpUvXKTUKTAamru8NRV8V0sYrn0pFlTO4zqV/xM+DAVpG7Getd+dJyUqPBwd5PkPE0MXYC1vDsyamNN2nZp6/O7R/PXzzDy7QaGPwkuKiykzTk5gAjlfyFZln0P6xmvE6eBQsY1ZQdnp+9+uHVmf3027Pm2bdPDfZhoSgOA43PrTWKNR1Jt2HHh5lZaENmMDcjhGPKKfaZZ1CHjqCjjFXOyPEWOScJNYy/On/0Q8BbjB3Uj2XgBZ6AkBziZd35ibAy+Gtcxe5pstRdRRrHeoCuDDRyi4nplqYBl4oxznZ2AVghF96w5cUUwaUzHBbnfL+57Z/9s3/2z/7ZP/tn/+yf/bN/9s/+2T/7Z//sn/2zf/bP/tk/+2f/7J/988Ce/wf0RFpLABgBAA=='
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('.')
sys.path.insert(0, W0); os.makedirs('results', exist_ok=True)
from vsearch.weights import discover
t=time.time(); Wt = discover('/kaggle/input', '/tmp/w'); Wt['assemble_s']=round(time.time()-t,1)
R['weights']={k:v for k,v in Wt.items() if k!='layout'}
ok = Wt.get('dinov3') and Wt.get('sam31_ckpt') and Wt.get('sam31_sha256_ok', True)
if not ok: R['errors'].append('weights missing / sha mismatch'); save(); raise SystemExit('stop')
os.environ['DINOV3_PATH']=Wt['dinov3']; os.environ['SAM3_CKPT']=Wt['sam31_ckpt']
run('nvidia_smi','nvidia-smi --query-gpu=name,memory.total --format=csv'); print(R['weights'])

In [ ]:
run('install','pip -q install faiss-cpu pycocotools psutil && git clone -q https://github.com/facebookresearch/sam3.git /tmp/sam3 && pip -q install -e /tmp/sam3 && cd /tmp/sam3 && git rev-parse HEAD', 900)
print(R['stages']['install'])

In [ ]:
run('data','wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /tmp/davis.zip && mkdir -p data && unzip -q /tmp/davis.zip -d data && rm /tmp/davis.zip && python scripts/prepare_davis.py && python scripts/make_hard_queries.py && python scripts/add_distractors.py', 1200)
print(R['stages']['data']['rc'], R['stages']['data'].get('sec'))

In [ ]:
run('gpu_bench','python scripts/gpu_bench.py', 1200)
try: R['gpu_bench']=json.load(open('results/gpu_bench.json'))
except Exception as e: R['errors'].append('gpu_bench json: %r'%e)
save(); print(R.get('gpu_bench'))

In [ ]:
B='data/davis_bench_large'; D=os.environ['DINOV3_PATH']
run('index_grid',f'python scripts/build_index.py --videos {B}/videos --out data/L_dinov3 --model {D} --regions grid', 1800)
run('index_sam3',f'python scripts/build_index.py --videos {B}/videos --out data/L_dinov3_sam3 --model {D} --regions sam3', 2700)
R['index']={}
for d in glob.glob('data/L_*'):
    try: m=json.load(open(d+'/meta.json')); m.pop('meta',None); R['index'][os.path.basename(d)]=m
    except Exception as e: R['errors'].append(f'index meta {d}: {e!r}')
save(); print(R['index'])

In [ ]:
R['eval']={}
for Q in ['object_hard','object']:
    for tag,cmd in [('dinov3_grid',f'--index data/L_dinov3'),('dinov3_sam3region',f'--index data/L_dinov3_sam3'),
                    ('dinov3_rerank_sam3_top20',f'--index data/L_dinov3 --modes both --rerank sam3 --videos {B}/videos')]:
        o=f'results/K_{tag}_{Q}.json'; run(f'eval_{tag}_{Q}', f'python scripts/evaluate.py {cmd} --queries {B}/queries_{Q}.json --out {o}', 2400)
        try: R['eval'][f'{tag}/{Q}']=json.load(open(o))['results']
        except Exception as e: R['errors'].append(f'eval {tag}/{Q}: {e!r}')
        save()
R['finished']=time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()); save()
print(json.dumps(R['eval'], indent=1)[:4000]); print('errors', R['errors'])
for f in os.listdir('/kaggle/working'):
    if f!='de12_results.json': subprocess.run(['rm','-rf','/kaggle/working/'+f])
print(os.listdir('/kaggle/working'))